<a href="https://colab.research.google.com/github/Akash9888/thesis_code/blob/master/sentiment_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"--- RUNNING ON {device.upper()} ---")


tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device)

if device == "cuda":
    model.half()

INPUT_FILE = '/content/drive/MyDrive/ThesisData/news_with_entities.csv'
OUTPUT_FILE = '/content/drive/MyDrive/ThesisData/final_enriched_news.csv'

def run_sentiment_analysis():
    df = pd.read_csv(INPUT_FILE)

    df['headline'] = df['headline'].astype(str).fillna("")
    headlines = df['headline'].tolist()

    sentiment_scores = []
    batch_size = 32

    print(f"Starting Sentiment Analysis on {len(df)} rows...")

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(headlines), batch_size)):
            batch = headlines[i:i+batch_size]


            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            scores = (probs[:, 0] - probs[:, 1]).cpu().numpy()
            sentiment_scores.extend(scores)

    df['sentiment_score'] = sentiment_scores
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nSUCCESS: Final enriched data saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    run_sentiment_analysis()

Mounted at /content/drive
--- RUNNING ON CPU ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting Sentiment Analysis on 199404 rows...


100%|██████████| 6232/6232 [5:26:46<00:00,  3.15s/it]



SUCCESS: Final enriched data saved to /content/drive/MyDrive/ThesisData/final_enriched_news.csv
